In [1]:
import spot

class SpotLTLMutator:
    # 1. Map Spot's built-in operator kinds to your category sets
    # spot.op_G (□), spot.op_F (♢), spot.op_X (◯), spot.op_Not (¬)
    UNARY_KINDS = {spot.op_G, spot.op_F, spot.op_X, spot.op_Not}
    
    # spot.op_Or (∨), spot.op_And (∧), spot.op_U (U), spot.op_R (R), spot.op_W (W)
    BINARY_KINDS = {spot.op_Or, spot.op_And, spot.op_U, spot.op_R, spot.op_W}
    
    # Allowed binary operators for Rule 3(d)
    RULE_3D_OPS = {spot.op_U, spot.op_W, spot.op_And, spot.op_Or}

    def __init__(self, atomic_propositions: list):
        """Initialize with a list of AP strings, e.g., ['p', 'q']"""
        self.ap_strings = set(atomic_propositions)
        self.ap_formulas = [spot.formula.ap(p) for p in atomic_propositions]
        self.constants = [spot.formula_ff(), spot.formula_tt()] # false, true

    def mutate(self, phi: spot.formula) -> list:
        """
        Main entry point. Takes a spot.formula object and returns 
        a deduplicated list of mutated spot.formula objects.
        """
        mutations = []
        phi_kind = phi.kind()

        # --- Structural Check Using Pure Explicit Kinds ---
        if phi_kind == spot.op_tt or phi_kind == spot.op_ff:
            mutations.extend(self._get_base_cases(phi, is_constant=True))
            
        elif phi_kind == spot.op_ap:
            mutations.extend(self._get_base_cases(phi, is_constant=False))
            
        elif phi_kind in self.UNARY_KINDS: 
            mutations.extend(self._get_unary_cases(phi))
            
        elif phi_kind in self.BINARY_KINDS: 
            mutations.extend(self._get_binary_cases(phi))

        # --- Apply General Cases (Rules 5 & 6) ---
        mutations.extend(self._get_general_cases(phi))

        # Deduplicate safely using unique string representations
        seen = set()
        deduped = []
        for m in mutations:
            f_str = m.to_str()
            if f_str not in seen:
                seen.add(f_str)
                deduped.append(m)
                
        return deduped

    def _get_base_cases(self, phi, is_constant: bool) -> list:
        """Handles Rules 1 and 2"""
        mutations = []
        if is_constant:
            # Rule 1: true -> false; false -> true
            mutations.append(spot.formula_ff() if phi.kind() == spot.op_tt else spot.formula_tt())
        else:
            # Rule 2: If p -> q where p != q
            p_str = phi.to_str()
            for q in self.ap_formulas:
                if q.to_str() != p_str:
                    mutations.append(q)
        return mutations

    def _get_unary_cases(self, phi: spot.formula) -> list:
        """Handles Rule 3 (Inductive Unary)"""
        mutations = []
        op1 = phi.kind()
        phi1 = phi[0]  # Safely get structural sub-operand
        
        # (a) phi' = op1' phi1
        for op1_prime in self.UNARY_KINDS:
            if op1 != op1_prime:
                mutations.append(spot.formula_unop(op1_prime, phi1))
                
        # (b) phi' = phi1
        mutations.append(phi1)
        
        # (c) phi' = op1 mutate(phi1)
        for m in self.mutate(phi1):
            mutations.append(spot.formula_unop(op1, m))
            
        # (d) phi' = p op2' phi
        for p in self.ap_formulas:
            for op2_prime in self.RULE_3D_OPS:
                mutations.append(spot.formula_binop(op2_prime, p, phi))
                
        return mutations

    def _get_binary_cases(self, phi: spot.formula) -> list:
        """Handles Rule 4 (Inductive Binary)"""
        mutations = []
        op2 = phi.kind()
        phi1 = phi[0]  # Left sub-operand
        phi2 = phi[1]  # Right sub-operand
        
        # (a) phi' = phi1 op2' phi2
        for op2_prime in self.BINARY_KINDS:
            if op2 != op2_prime:
                mutations.append(spot.formula_binop(op2_prime, phi1, phi2))
                
        # (b) phi' = phi_i
        mutations.append(phi1)
        mutations.append(phi2)
        
        # (c) phi' = mutate(phi1) op2 phi2
        for m in self.mutate(phi1):
            mutations.append(spot.formula_binop(op2, m, phi2))
            
        # (d) phi' = phi1 op2 mutate(phi2)
        for m in self.mutate(phi2):
            mutations.append(spot.formula_binop(op2, phi1, m))
            
        return mutations

    def _get_general_cases(self, phi: spot.formula) -> list:
        """Handles Rules 5 and 6 (General Cases)"""
        mutations = []
        
        # Rule 5: wrap the whole formula in a unary operator
        for op1 in self.UNARY_KINDS:
            mutations.append(spot.formula_unop(op1, phi))
            
        # Rule 6: Replace with true, false, or any AP
        mutations.extend(self.constants)
        mutations.extend(self.ap_formulas)
        
        return mutations


# --- Example Tester Verification ---
if __name__ == "__main__":
    mutator = SpotLTLMutator(atomic_propositions=["p", "q"])
    
    # Parse a test formula string cleanly using Spot syntax
    input_formula = spot.formula("G(p | F q)")
    
    print(f"Original Spot Formula: {input_formula}")
    print("-" * 50)
    
    results = mutator.mutate(input_formula)
    print(f"Generated {len(results)} distinct mutants via Spot API:\n")
    
    for m in sorted(results, key=lambda x: x.to_str())[:15]: 
        print(f"  -> {m.to_str()}")

: 

In [ ]:
{spot.op_G, spot.op_F, spot.op_X, spot.op_Not}
{spot.op_Or, spot.op_And, spot.op_U, spot.op_R, spot.op_W}
{spot.op_U, spot.op_W, spot.op_And, spot.op_Or}

{14, 16, 21, 23}